# Big Data Analytics: 2-Hour Self-Tutorial

**Estimated Completion Time:** 2 Hours

This tutorial bridges the theoretical concepts of Big Data storage, the MapReduce paradigm, and algebraic operations into practical, executable code. You will build parallel processing pipelines from scratch and transition into industry-standard tools like Apache Spark.

### Learning Objectives:
1. Understand and implement the MapReduce architecture natively.
2. Utilize Apache Spark (PySpark) for parallel algebraic operations on RDDs.
3. Process structured data using Spark DataFrames.
4. Explore Graph Data structures.
5. Prepare your environment for downstream Deep Learning tasks.

---
## Part 1: The MapReduce Paradigm (Core Concepts)

Before relying on distributed frameworks, we must understand the core mechanics of `map()` and `reduce()`. In a distributed system, the `map` function processes input key-value pairs to generate intermediate pairs. The `shuffle` phase groups these by key, and the `reduce` function aggregates them.

### Exercise: Native Python Word Count
Review the skeleton and complete the `reduce_function`.

In [1]:
from collections import defaultdict

# Sample distributed file blocks
documents = [
    "traditional applications of relational databases",
    "modern data management applications need data",
    "big data refers to processing of large data"
]

def map_function(document):
    """Emits (word, 1) for each word in the document."""
    results = []
    for word in document.split():
        results.append((word, 1))
    return results

def shuffle_function(mapped_values):
    """Groups emitted values by key."""
    grouped_data = defaultdict(list)
    for key, value in mapped_values:
        grouped_data[key].append(value)
    return grouped_data

def reduce_function(key, values):
    """Aggregates the list of values for a given key."""
    # task sol 1
    total_count = 0
    for value in values:
        total_count += value
    return (key, total_count)

    # 2nd way
    # total_count = sum(values)

# Simulating the MapReduce Pipeline
print("Running Native MapReduce...")

# 1. Map Phase
mapped_results = []
for doc in documents:
    mapped_results.extend(map_function(doc))

# 2. Shuffle Phase
shuffled_data = shuffle_function(mapped_results)

# 3. Reduce Phase
final_output = []
for key, values in shuffled_data.items():
    final_output.append(reduce_function(key, values))

# Displaying top 5 results sorted by frequency
final_output.sort(key=lambda x: x[1], reverse=True)
for record in final_output[:5]:
    print(record)

Running Native MapReduce...
('data', 4)
('applications', 2)
('of', 2)
('traditional', 1)
('relational', 1)


---
## Part 2: Apache Spark and Algebraic Operations

Native Python does not scale to thousands of machines. Apache Spark provides an engine for large-scale data processing using Resilient Distributed Datasets (RDDs). Spark performs operations lazily, building a Directed Acyclic Graph (DAG) of transformations before executing them via an action.

In [2]:
# Install PySpark in the Colab Environment
%pip install pyspark -q

from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("BigDataTutorial") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark Context Initialized Successfully.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Spark Context Initialized Successfully.


### Exercise: Inverted Index with Spark
An inverted index is crucial for search engines. It maps keywords to the documents (or document IDs) where they appear.

In [3]:
# Input data: (DocumentID, Text)
data = [
    (1, "data clean"),
    (2, "data base"),
    (3, "clean base")
]

rdd = sc.parallelize(data)

# TODO: Implement the map and reduce operations to create an inverted index.
# Step 1: Map to (word, document_id)
def map_to_index(record):
    doc_id, text = record
    return [(word, doc_id) for word in text.split()]

mapped_rdd = rdd.flatMap(map_to_index)

# Step 2: Group by word, and convert the grouped document IDs into a unique list or string
inverted_index_rdd = mapped_rdd.groupByKey().mapValues(lambda doc_ids: list(set(doc_ids)))

# Execute and show results
results = inverted_index_rdd.collect()
for word, docs in results:
    print(f"Word: '{word}' -> Found in Docs: {docs}")

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 4 in stage 0.0 failed 1 times, most recent failure: Lost task 4.0 in stage 0.0 (TID 4) (DESKTOP-GJTQAQJ.mshome.net executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:281)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:309)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:72)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.api.python.PairwiseRDD.compute(PythonRDD.scala:140)
	at

---
## Part 3: Advanced RDD Operations (Word Pairs)
Analyzing the relationship between consecutive words is a foundational step in Natural Language Processing (NLP) and recommendation systems.

In [ ]:
text_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the fast brown fox is quick",
    "a quick brown dog"
]

docs_rdd = sc.parallelize(text_corpus)

# TODO: Extract consecutive word pairs and count their occurrences
def extract_pairs(line):
    words = line.lower().split()
    pairs = []
    for i in range(len(words) - 1):
        pairs.append((f"{words[i]} {words[i+1]}", 1))
    return pairs

pairs_rdd = docs_rdd.flatMap(extract_pairs)
pair_counts_rdd = pairs_rdd.reduceByKey(lambda a, b: a + b)

# Filter to find pairs that appear more than once
frequent_pairs = pair_counts_rdd.filter(lambda x: x[1] > 1).collect()

print("Frequent Word Pairs:")
for pair, count in frequent_pairs:
    print(f"{pair}: {count}")

---
## Part 4: Structured Data with Spark DataFrames
Modern Big Data relies heavily on structured formats like Parquet or JSON. Spark DataFrames provide a higher-level API over RDDs, allowing for SQL-like algebraic transformations.

In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, desc

# Simulating loading a structured dataset (e.g., from a Parquet file)
instructors = [
    Row(ID="101", name="Katz", dept_name="Comp. Sci.", salary=75000),
    Row(ID="102", name="Einstein", dept_name="Physics", salary=120000),
    Row(ID="103", name="Turing", dept_name="Comp. Sci.", salary=110000)
]

df = spark.createDataFrame(instructors)
df.show()

# Algebraic Query: Find Computer Science instructors with salary > 80000
high_earners = df.filter((col("dept_name") == "Comp. Sci.") & (col("salary") > 80000))
print("High earning CS Instructors:")
high_earners.show()

---
## Part 5: Hardware Configuration for Downstream Deep Learning
Big data processing pipelines often serve as the data ingestion layer for Deep Learning models. When transitioning from data processing to model training, properly configuring hardware acceleration is essential.

In [ ]:
import torch

# Standard configuration for deep learning device assignment
device = ('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

print("Computation device set to:", device)

# Cleanup Spark Session
spark.stop()